# Клип «А представь, мы станем грибами» — липсинк + сборка 1920×1080

**Что делает ноутбук:** скачивает твои клипы и песню, режет на 25 слотов, **прогоняет 18 слотов с лицом через LatentSync** (рот подгоняется под настоящий голос — под тот кусок песни, который звучит именно в этом месте), собирает финал **1920×1080** со звуком и складывает в `/kaggle/working`.

**Как запустить в Kaggle:**
1. `File → Import Notebook → GitHub` → вставь ссылку на этот файл.
2. Справа в панели **Settings**:
   - **Accelerator: GPU T4 ×2** (или P100)
   - **Internet: On** — обязательно, без него не скачаются модель и клипы
3. Нажми **Run All**. Всё.

**Время:** установка ~6 мин, липсинк ~25–35 мин, сборка ~6 мин. Готовый файл появится в панели справа (Output) — его можно скачать.


In [ ]:
#@title 1. Проверка GPU и интернета
import torch, subprocess, urllib.request
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'Включи Settings → Accelerator → GPU T4 x2 и перезапусти.'
try:
    urllib.request.urlopen('https://huggingface.co', timeout=15)
    print('интернет: ок')
except Exception:
    raise SystemExit('Включи Settings → Internet → On, затем запусти ячейку снова.')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'ГБ')


In [ ]:
#@title 2. LatentSync + веса (6 минут)
import os, subprocess, torch, shutil
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/LatentSync'):
    subprocess.run(['git','clone','--quiet','--depth','1','https://github.com/bytedance/LatentSync.git'], check=True)
os.chdir('/kaggle/working/LatentSync')
subprocess.run(['pip','-q','install','gdown','imageio-ffmpeg','--no-warn-script-location'], check=True)

vram = torch.cuda.get_device_properties(0).total_memory/1e9
VER = '1.6' if vram >= 18 else '1.5'
CFG = 'configs/unet/stage2_512.yaml' if VER == '1.6' else 'configs/unet/stage2.yaml'
print(f'{vram:.1f} ГБ видеопамяти → LatentSync {VER}')

need = ['diffusers==0.32.2','transformers==4.48.0','accelerate==0.26.1','einops==0.7.0','omegaconf==2.3.0',
        'opencv-python-headless==4.9.0.80','mediapipe==0.10.11','python_speech_features==0.6','librosa==0.10.1',
        'scenedetect==0.6.1','ffmpeg-python==0.2.0','lpips==0.1.4','face-alignment==1.4.1','huggingface-hub==0.30.2',
        'kornia==0.8.0','onnxruntime-gpu==1.21.0','DeepCache==0.1.1','numpy==1.26.4','decord==0.6.0']
subprocess.run(['pip','-q','install',*need,'--no-warn-script-location'], check=False)

from huggingface_hub import hf_hub_download
os.makedirs('checkpoints', exist_ok=True)
for f in ['whisper/tiny.pt','latentsync_unet.pt']:
    hf_hub_download(f'ByteDance/LatentSync-{VER}', f, local_dir='checkpoints')
print('готово:', os.listdir('checkpoints'))


In [ ]:
#@title 3. Клипы и песня с Google Drive
import subprocess, glob, os
if not os.path.exists('/kaggle/working/song.mp3'):
    subprocess.run(['gdown','1yGs88oWAw6TP-EdskGqH41Sjc2m0M309','-O','/kaggle/working/song.mp3'], check=True)
srcs = glob.glob('/kaggle/working/src/**/*_alligator.mp4', recursive=True)
if len(srcs) < 26:
    subprocess.run(['gdown','--folder','https://drive.google.com/drive/folders/1Vj111PVH8__jcgNrqI872KOtB2sUqOR6',
                    '-O','/kaggle/working/src'], check=False)
    srcs = glob.glob('/kaggle/working/src/**/*_alligator.mp4', recursive=True)
print('клипов:', len(srcs), '| песня:', round(os.path.getsize('/kaggle/working/song.mp3')/1e6,1),'МБ')
assert len(srcs) >= 24, 'Клипы не скачались — проверь Settings → Internet → On'


In [ ]:
#@title 4. Нарезка слотов (4 минуты)
import os, subprocess, glob, shutil, imageio_ffmpeg
FF = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
EDL = [{"sheet": 1, "clip": "b7g6AO-SsLyl-1HMYHR94", "pieces": [[0.35, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 2, "clip": "F05dTWcv-J38JotcdOFkb", "pieces": [[2.6, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 3, "clip": "vsjhjHwbk2iBp2-wO3uDy", "pieces": [[4.15, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 4, "clip": "C6CS0iMi2wicpnbJKPiYO", "pieces": [[2.45, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 5, "clip": "6-Y17BE-dYXXB1xv-UHeV", "pieces": [[0.4, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 6, "clip": "jGxBAGohUw_Ynz1OIMuw4", "pieces": [[0.4, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 7, "clip": "I70IjbuFvOxoNIawdpEby", "pieces": [[1.05, 8.2], [9.75, 1.3]], "dur": 10.0, "speed": 0.95}, {"sheet": 8, "clip": "C0yjn0ycpc6AMvTAspvYu", "pieces": [[1.05, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 9, "clip": "AUg40zILdpFzNqpplCA6X", "pieces": [[1.25, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 10, "clip": "Tkejtv3_jGQzXLlkKKlSS", "pieces": [[1.05, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 11, "clip": "BKa1owgsawSmdNkTLPhQI", "pieces": [[0.65, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 12, "clip": "nnleYkMHWXzHGxiL843vL", "pieces": [[0.85, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 13, "clip": "0TmK7dfF4nHyXvB22iQB7", "pieces": [[0.65, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 14, "clip": "vpW18Vw_2FhtJ2SU34tiy", "pieces": [[0.65, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 15, "clip": "8u5YqxnHC4V3kGAkquqLT", "pieces": [[5.8, 9.15]], "dur": 10.0, "speed": 0.915}, {"sheet": 16, "clip": "lNr9rOsborrH_CNtKZ0J9", "pieces": [[0.65, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 17, "clip": "2d3CTKgfZfxudUO9_ohp4", "pieces": [[1.4, 2.3], [5.4, 3.5]], "dur": 10.0, "speed": 0.85}, {"sheet": 18, "clip": "5YlImGBwtmVVl4VrApLcA", "pieces": [[0.8, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 19, "clip": "TWxeljKSqtHQ2efheHfn9", "pieces": [[0.45, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 20, "clip": "mNttoqYxIhAJcGavJJ784", "pieces": [[0.6, 3.1], [9.8, 5.15]], "dur": 10.0, "speed": 0.85}, {"sheet": 21, "clip": "4_QcHFF_-7DT9UXBk8BGo", "pieces": [[1.05, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 22, "clip": "55-m8In0chfWOHjS4T30j", "pieces": [[0.6, 7.45], [10.2, 2.55]], "dur": 10.0, "speed": 1.0}, {"sheet": 23, "clip": "qQA9zjJ_TIIxYuxwFUDUO", "pieces": [[3.2, 1.5], [5.8, 8.5]], "dur": 10.0, "speed": 1.0}, {"sheet": 24, "clip": "ODrVP7atbvsNrlkRo7jbt", "pieces": [[2.6, 10.0]], "dur": 10.0, "speed": 1.0}, {"sheet": 25, "clip": "wNcHdwJ6NWgoaa34-C5Hr", "pieces": [[3.05, 9.48]], "dur": 9.48, "speed": 1.0}]
SRC = {os.path.basename(p).replace('_alligator.mp4',''): p for p in glob.glob('/kaggle/working/src/**/*_alligator.mp4', recursive=True)}
SEG = '/kaggle/working/segs'; os.makedirs(SEG, exist_ok=True)
def run(a, **k): return subprocess.run(a, check=True, capture_output=True, **k)
for i, s in enumerate(EDL):
    t0 = round(10.0*i, 3); parts = []
    for k,(st,take) in enumerate(s['pieces']):
        out = f'{SEG}/s{i:02d}_p{k}.mp4'
        run([FF,'-y','-ss',str(st),'-t',str(take),'-i',SRC[s['clip']],'-an',
             '-vf','fps=25,scale=-2:1080','-c:v','libx264','-crf','14','-preset','veryfast',out])
        parts.append(out)
    raw = parts[0]
    if len(parts) > 1:
        lst = f'{SEG}/s{i:02d}.txt'; open(lst,'w').write(''.join(f"file '{p}'\n" for p in parts))
        raw = f'{SEG}/s{i:02d}_raw.mp4'
        run([FF,'-y','-f','concat','-safe','0','-i',lst,'-c','copy',raw])
    run([FF,'-y','-i',raw,'-vf',f"setpts=(PTS-STARTPTS)/{s['speed']:.4f},fps=25,tpad=stop_mode=clone:stop_duration=1.0",
         '-t',f"{s['dur']+0.6:.2f}",'-an','-c:v','libx264','-crf','14','-preset','veryfast',f'{SEG}/s{i:02d}.mp4'])
    run([FF,'-y','-ss',f'{t0:.3f}','-t',f"{s['dur']:.3f}",'-i','/kaggle/working/song.mp3','-ar','16000','-ac','1',f'{SEG}/a{i:02d}.wav'])
    print(f'слот {i+1:02d} (лист {s["sheet"]:02d}) готов', end='  ')
print('\nнарезка завершена')


In [ ]:
#@title 5. Липсинк по слотам с лицом (25–35 минут)
import os, shutil, subprocess
FACE = set([1, 4, 6, 7, 8, 9, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25])
for i, s in enumerate(EDL):
    src, aud, out = f'{SEG}/s{i:02d}.mp4', f'{SEG}/a{i:02d}.wav', f'{SEG}/sync{i:02d}.mp4'
    if os.path.exists(out) and os.path.getsize(out) > 10000:
        print(f'{i+1:02d} уже готов'); continue
    if s['sheet'] not in FACE:
        shutil.copy(src, out); print(f'{i+1:02d} без лица'); continue
    base = ['python','-m','scripts.inference','--unet_config_path',CFG,
            '--inference_ckpt_path','checkpoints/latentsync_unet.pt',
            '--inference_steps','20','--guidance_scale','1.5',
            '--video_path',src,'--audio_path',aud,'--video_out_path',out]
    ok = False
    for extra in (['--enable_deepcache'], []):
        subprocess.run(base+extra, cwd='/kaggle/working/LatentSync', capture_output=True, text=True)
        if os.path.exists(out) and os.path.getsize(out) > 10000:
            ok = True; break
    if not ok:
        shutil.copy(src, out); print(f'{i+1:02d} не удалось, оставил оригинал')
    else:
        print(f'{i+1:02d} липсинк готов')
print('липсинк завершён')


In [ ]:
#@title 6. Сборка 1920×1080 (6 минут)
import os, subprocess, shutil, imageio_ffmpeg
FF = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
N = len(EDL); DISS = 0.30
T = [0.0]
for s in EDL: T.append(T[-1] + s['dur'])
OUT = '/kaggle/working/КЛИП_Грибы_1080p_липсинк.mp4'
for i in range(N):
    subprocess.run([FF,'-y','-i',f'{SEG}/sync{i:02d}.mp4','-vf',
        'fps=24,scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080,setsar=1,'
        'tpad=stop_mode=clone:stop_duration=1.0','-an','-c:v','libx264','-preset','fast','-crf','15',
        f'{SEG}/f{i:02d}.mp4'], check=True)
for ci,(a,b) in enumerate([(1,13),(13,26)]):
    ins = []
    for k in range(a-1,b-1): ins += ['-i', f'{SEG}/f{k:02d}.mp4']
    fc, prev = [], '[0:v]'
    for k in range(a,b-1):
        off = (T[k]-T[a-1]) - DISS
        fc.append(f'{prev}[{k-a+1}:v]xfade=transition=fade:duration={DISS}:offset={off:.3f}[v{k}]')
        prev = f'[v{k}]'
    dur_blk = round(T[b-1]-T[a-1], 3)
    if ci == 0:
        fc.append(f'{prev}fade=t=out:st={dur_blk-0.18:.3f}:d=0.18,format=yuv420p[v]')
    else:
        fc.append(f'{prev}fade=t=in:st=0:d=0.18,fade=t=out:st={dur_blk-1.6:.3f}:d=1.6,format=yuv420p[v]')
    subprocess.run([FF,'-v','error','-y']+ins+['-filter_complex',';'.join(fc),'-map','[v]',
        '-c:v','libx264','-preset','slow','-crf','17','-pix_fmt','yuv420p','-r','24','-an',
        '-t',f'{dur_blk:.3f}',f'/kaggle/working/prt{ci}.mp4'], check=True)
    print('блок', ci+1, 'готов')
open('/kaggle/working/prt.txt','w').write("file '/kaggle/working/prt0.mp4'\nfile '/kaggle/working/prt1.mp4'\n")
subprocess.run([FF,'-v','error','-y','-f','concat','-safe','0','-i','/kaggle/working/prt.txt',
    '-i','/kaggle/working/song.mp3','-map','0:v','-map','1:a','-c:v','copy','-c:a','aac','-b:a','320k',
    '-ar','48000','-af','afade=t=out:st=247.2:d=2.2','-movflags','+faststart','-t','249.480',OUT], check=True)
print('ГОТОВО:', OUT, round(os.path.getsize(OUT)/1e6,1),'МБ — скачай справа в Output')


## Если что-то не так
- **Нет GPU** — Settings → Accelerator → GPU T4 ×2.
- **Не качается модель** — Settings → Internet → On, перезапусти ячейку 2.
- **Упала ячейка 5** — просто запусти её снова: уже сделанные слоты пропустятся.
- **Не хватает диска** — 20 ГБ хватает с запасом; промежуточные файлы в `/kaggle/working/segs` можно удалить после сборки.
